# 🎸 Music2Chart — Gerador Automático de Charts (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adiells/music2chart/blob/main/music2chart_colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Music2Chart-181717?logo=github)](https://github.com/Adiells/music2chart)
[![Web Studio](https://img.shields.io/badge/Web_Studio-music2chart.adiels.me-00f5d4?logo=googlechrome&logoColor=black)](https://music2chart.adiels.me/)

Converta qualquer música (**MP3, WAV, FLAC, OGG**) ou link do YouTube em charts de nível torneio para **Clone Hero** e **YARG** (`.chart` e `.sng`) utilizando a GPU gratuita do Google Colab!

### ⚡ Destaques da Pipeline:
- **Separação de Guitarras**: Meta Demucs v4 (htdemucs) acelerado por CUDA.
- **Transcrição Neural**: Spotify Basic Pitch para identificação polifônica precisa de notas e acordes.
- **Heurísticas de Game Design**: Grade rítmica a 192 PPQ, tessitura dinâmica em 5 trastes, HOPOs automáticos e frases de Star Power.
- **Empacotamento Instantâneo**: Geração automática de `notes.chart`, `song.ini`, áudio e container `.sng` em um arquivo `.zip` pronto para jogar.

## 1. Verificar Aceleração por GPU
Certifique-se de que o Colab está conectado a um ambiente com GPU:
> Menu superior: **Ambiente de execução (Runtime)** ➔ **Alterar tipo de ambiente de execução** ➔ Selecione **T4 GPU**.

In [ ]:
#@title 1. Testar GPU CUDA
import torch

print('=' * 60)
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'✅ GPU Ativa: {device_name} ({vram_gb:.1f} GB VRAM)')
    print('Aceleração por hardware configurada com sucesso!')
else:
    print('⚠️ Nenhuma GPU detectada. O processamento rodará na CPU (mais lento).')
    print("Recomendado: Vá em 'Ambiente de Execução' -> 'Alterar tipo de ambiente' -> Selecione 'T4 GPU'.")
print('=' * 60)


## 2. Instalar Music2Chart e Dependências
Clona o repositório oficial e instala as bibliotecas necessárias de inteligência artificial e processamento de áudio.

In [ ]:
#@title 2. Instalar Music2Chart e Ferramentas
import os
import sys

if not os.path.exists('music2chart'):
    !git clone --depth 1 https://github.com/Adiells/music2chart.git

%cd music2chart

print('📦 Instalando dependências de IA e áudio (leva ~1 minuto)...')
!pip install -q --upgrade pip
!pip install -q 'setuptools<72.0.0'
!pip install -q basic-pitch demucs librosa soundfile scipy numpy python-multipart mido tqdm yt-dlp
!pip install -q -e .

print('✅ Ambiente Music2Chart configurado com sucesso!')


## 3. Configurar Música e Presets
Preencha as informações da música e escolha o perfil de geração mais adequado.

In [ ]:
#@title 3. Configurações da Música e Preset
#@markdown ### 🎵 Origem do Áudio
AUDIO_SOURCE = "Upload Manual" #@param ["Upload Manual", "Link do YouTube", "Arquivo Local no Colab"]
YOUTUBE_URL = "https://www.youtube.com/watch?v=..." #@param {type:"string"}
LOCAL_FILE_PATH = "/content/sua_musica.mp3" #@param {type:"string"}

#@markdown ### 🏷️ Metadados da Música
SONG_NAME = "Sweet Child O Mine" #@param {type:"string"}
ARTIST_NAME = "Guns N Roses" #@param {type:"string"}

#@markdown ### 🎛️ Preset de Calibração
#@markdown - `the_best`: *(Recomendado)* Noise gate 0.41, escala BPM 0.65, quantização estrita e fusão sólida de acordes.
#@markdown - `standard`: Mapeamento balanceado em 5 trastes para rock e metal.
#@markdown - `strict_rhythm`: Quantização rígida com ênfase em acordes e base rítmica.
#@markdown - `dynamic_solo`: Para solos de alta velocidade, sweeps e frases virtuosas.
PRESET = "the_best" #@param ["the_best", "standard", "strict_rhythm", "dynamic_solo"]

#@markdown ### 📦 Opções de Saída
OUTPUT_FORMAT = "both" #@param ["both", "chart", "sng"]
DIFFICULTY = 6 #@param {type:"slider", min:1, max:6, step:1}

print(f'🎵 Música: {SONG_NAME} - {ARTIST_NAME}')
print(f'🎛️ Preset: {PRESET} | Dificuldade: {DIFFICULTY} | Formato: {OUTPUT_FORMAT}')


## 4. Obter Áudio para Processamento
Faz upload do seu arquivo de áudio ou baixa diretamente a faixa pelo YouTube.

In [ ]:
#@title 4. Carregar Áudio
import os
import shutil
from pathlib import Path
from google.colab import files

INPUT_DIR = Path('/content/input_audio')
INPUT_DIR.mkdir(parents=True, exist_ok=True)
audio_input_path = None

if AUDIO_SOURCE == "Upload Manual":
    print('📤 Clique no botão abaixo para selecionar seu áudio (MP3, WAV, FLAC, OGG):')
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('Nenhum arquivo enviado. Por favor, envie uma faixa de áudio.')
    uploaded_filename = next(iter(uploaded.keys()))
    audio_input_path = INPUT_DIR / uploaded_filename
    shutil.move(uploaded_filename, audio_input_path)
    print(f'✅ Arquivo carregado: {audio_input_path.name}')

elif AUDIO_SOURCE == "Link do YouTube":
    if not YOUTUBE_URL or ('youtube.com' not in YOUTUBE_URL and 'youtu.be' not in YOUTUBE_URL):
        raise ValueError('Por favor, informe uma URL válida do YouTube.')
    print(f'⬇️ Baixando áudio do YouTube: {YOUTUBE_URL}...')
    import yt_dlp
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': str(INPUT_DIR / '%(title)s.%(ext)s'),
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'quiet': False
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(YOUTUBE_URL, download=True)
        filename = ydl.prepare_filename(info)
        base, _ = os.path.splitext(filename)
        audio_input_path = Path(f'{base}.mp3')
        if SONG_NAME == 'Sweet Child O Mine' and info.get('title'):
            SONG_NAME = info.get('title')
        if ARTIST_NAME == 'Guns N Roses' and info.get('uploader'):
            ARTIST_NAME = info.get('uploader')
    print(f'✅ Áudio extraído do YouTube: {audio_input_path.name}')

elif AUDIO_SOURCE == "Arquivo Local no Colab":
    audio_input_path = Path(LOCAL_FILE_PATH)
    if not audio_input_path.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {LOCAL_FILE_PATH}')
    print(f'✅ Arquivo selecionado: {audio_input_path.name}')

print(f'🎯 Pronto para processamento: {audio_input_path}')


## 5. Gerar Chart com IA
Executa o pipeline completo: separação por Demucs v4, transcrição polifônica Basic Pitch e mapeamento nos trastes do Clone Hero.

In [ ]:
#@title 5. Executar Music2Chart Pipeline
import sys
from pathlib import Path
from src.pipeline import Music2ChartPipeline

OUTPUT_DIR = Path('/content/output_charts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def colab_progress(ratio: float, stage: str):
    percent = int(ratio * 100)
    bar = '█' * (percent // 5) + '░' * (20 - (percent // 5))
    print(f'\r[{bar}] {percent:3d}% — {stage}', end='', flush=True)

print('🚀 Iniciando pipeline neural do Music2Chart...')
print('-' * 60)

pipeline = Music2ChartPipeline()
results = pipeline.process_song(
    audio_path=audio_input_path,
    song_name=SONG_NAME,
    artist=ARTIST_NAME,
    preset=PRESET,
    output_format=OUTPUT_FORMAT,
    output_dir=OUTPUT_DIR,
    difficulty=DIFFICULTY,
    progress_callback=colab_progress,
)

print('\n' + '-' * 60)
print('🎉 Chart gerado com sucesso!')
package_dir = results.get('package_dir')
print(f'📁 Pasta dos arquivos: {package_dir}')
for key, p in results.items():
    if p and Path(p).exists():
        print(f'   • {key.upper()}: {Path(p).name}')


## 6. Baixar Pacote (.ZIP) Pronto para Jogar
Baixa automaticamente o arquivo `.zip` para o seu computador.

In [ ]:
#@title 6. Baixar Arquivo .ZIP
import shutil
from pathlib import Path
from google.colab import files

package_path = Path(results['package_dir'])
zip_name = package_path.name
zip_archive = Path(f'/content/{zip_name}.zip')

print(f'📦 Compactando {package_path.name} em .zip...')
shutil.make_archive(str(zip_archive.with_suffix('')), 'zip', package_path)

print(f'⬇️ Iniciando download de {zip_archive.name} ({zip_archive.stat().st_size / (1024*1024):.2f} MB)...')
files.download(str(zip_archive))

print('=' * 60)
print('🎮 Como jogar no Clone Hero / YARG:')
print('1. Descompacte o arquivo .zip baixado.')
print('2. Copie a pasta da música para a pasta de músicas do jogo:')
print('   - Windows: Documentos\\Clone Hero\\Songs\\')
print('   - Mac: ~/Library/Application Support/Clone Hero/Songs/')
print('   - Linux: ~/.clonehero/Songs/')
print('3. Abra o jogo, vá em Configurações -> Geral -> Escanear Músicas (Scan Songs).')
print('4. Divirta-se!')
print('=' * 60)
